# Invoice & Payment Analysis

In [10]:
import pandas as pd
import numpy as np

# 1. Load the actual files
FILE_INVOICES = '/content/sample_data/Invoices.xlsx'
FILE_PAYMENTS = '/content/sample_data/Payments.xlsx'

# 1 & 2. Load the files and filter for SUCCESSFUL payments only
invoices = pd.read_excel(FILE_INVOICES, sheet_name='Invoices_Clean')
payments = pd.read_excel(FILE_PAYMENTS)

# Convert dates to datetime
invoices['invoice_date'] = pd.to_datetime(invoices['invoice_date'])
payments['created_at_ist'] = pd.to_datetime(payments['created_at_ist'])

# CRITICAL FIX: Only consider payments that actually went through ('captured')
successful_payments = payments[payments['status'] == 'captured'].copy()

# 3. Merge datasets using the successful payments
merged_df = pd.merge(
    invoices,
    successful_payments,
    left_on='invoice_id',
    right_on='invoice_ref',
    how='outer',
    indicator=True,
    suffixes=('_inv', '_pay')
)
print("Data loaded and merged successfully. Ready for analysis.")

Data loaded and merged successfully. Ready for analysis.


### 1. All the invoices without payment and total amount

In [12]:
invoices_no_payment = merged_df[merged_df['_merge'] == 'left_only']
total_unpaid = invoices_no_payment['amount_rupees_inv'].sum()

print(f"Total Invoices Without Payment: {len(invoices_no_payment)}")
print(f"Total Amount: ₹{total_unpaid:,.2f}")
display(invoices_no_payment[['invoice_id', 'invoice_date', 'amount_rupees_inv']].head())

Total Invoices Without Payment: 17256
Total Amount: ₹27,580,308.00


,invoice_id,invoice_date,amount_rupees_inv
35,INV-2025-000032,2025-01-01,720
49,INV-2025-000046,2025-01-01,1050
58,INV-2025-000055,2025-01-01,1600
87,INV-2025-000084,2025-01-01,800
105,INV-2025-000102,2025-01-01,720


### 2. All the payments without invoice and total amount

In [13]:
payments_no_invoice = merged_df[merged_df['_merge'] == 'right_only']
total_unmapped_payments = payments_no_invoice['amount_rupees_pay'].sum()

print(f"Total Payments Without Invoice: {len(payments_no_invoice)}")
print(f"Total Amount: ₹{total_unmapped_payments:,.2f}")
display(payments_no_invoice[['payment_id', 'created_at_ist', 'amount_rupees_pay']].head())

Total Payments Without Invoice: 0
Total Amount: ₹0.00


,payment_id,created_at_ist,amount_rupees_pay


### 3. All the invoices with payment done after the cutoff date of 30-06-2026 and total amount

In [11]:
### 3. All the invoices with payment done after the cutoff date of 30-06-2026 and total amount
cutoff_date = pd.to_datetime('2026-06-30')

# Filter for matched records where the payment date is strictly after the cutoff
late_payments_merged = merged_df[(merged_df['_merge'] == 'both') & (merged_df['created_at_ist'] > cutoff_date)]

# CRITICAL FIX: Drop duplicate invoice IDs so we don't double-count the invoice amount
# in cases where a single invoice had multiple late payments.
unique_late_invoices = late_payments_merged.drop_duplicates(subset=['invoice_id'])
total_late_amount = unique_late_invoices['amount_rupees_inv'].sum()

print(f"Unique Invoices with payment after 30-06-2026: {len(unique_late_invoices)}")
print(f"Total Invoice Amount: ₹{total_late_amount:,.2f}")
display(unique_late_invoices[['invoice_id', 'invoice_date', 'created_at_ist', 'amount_rupees_inv']].head())

Unique Invoices with payment after 30-06-2026: 2227
Total Invoice Amount: ₹3,728,402.00


,invoice_id,invoice_date,created_at_ist,amount_rupees_inv
458627,INV-2026-148815,2026-06-15,2026-06-30 02:16:13,1750
458642,INV-2026-148830,2026-06-15,2026-06-30 01:01:35,450
458658,INV-2026-148846,2026-06-15,2026-06-30 00:51:27,700
458749,INV-2026-148936,2026-06-15,2026-06-30 02:12:23,1799
458801,INV-2026-148988,2026-06-15,2026-06-30 02:25:17,1680


### 4. All the invoices and total amount with date before 24th June and no payment done

In [14]:
early_date = pd.to_datetime('2026-06-24')

# Filter the already-found unpaid invoices for dates before June 24th
early_unpaid_invoices = invoices_no_payment[invoices_no_payment['invoice_date'] < early_date]
total_early_unpaid_amount = early_unpaid_invoices['amount_rupees_inv'].sum()

print(f"Unpaid invoices dated before 24-06-2026: {len(early_unpaid_invoices)}")
print(f"Total Amount: ₹{total_early_unpaid_amount:,.2f}")
display(early_unpaid_invoices[['invoice_id', 'invoice_date', 'amount_rupees_inv']].head())

Unpaid invoices dated before 24-06-2026: 13547
Total Amount: ₹21,346,753.00


,invoice_id,invoice_date,amount_rupees_inv
35,INV-2025-000032,2025-01-01,720
49,INV-2025-000046,2025-01-01,1050
58,INV-2025-000055,2025-01-01,1600
87,INV-2025-000084,2025-01-01,800
105,INV-2025-000102,2025-01-01,720


### 5. All the places and total amount where duplicate payments are done

In [15]:
# 5. All the places and total amount where duplicate payments are done

# Remove rows where invoice_ref is blank
valid_payments = payments.dropna(subset=['invoice_ref'])

# Use keep='first' to ONLY flag the extra/duplicate payments (ignores the 1st valid payment)
extra_duplicate_payments = valid_payments[valid_payments.duplicated(subset=['invoice_ref'], keep='first')]

# Calculate the strictly excess amount paid
total_excess_amount = extra_duplicate_payments['amount_rupees'].sum()

print(f"Total Extra Duplicate Payment Transactions: {len(extra_duplicate_payments)}")
print(f"Total Excess Amount Collected: ₹{total_excess_amount:,.2f}")

# Display a sample of these extra payments
display(extra_duplicate_payments[['payment_id', 'invoice_ref', 'created_at_ist', 'amount_rupees']].sort_values('invoice_ref').head())

Total Extra Duplicate Payment Transactions: 19987
Total Excess Amount Collected: ₹31,126,241.00


,payment_id,invoice_ref,created_at_ist,amount_rupees
15,pay_0000000010,INV-2025-000011,2025-01-01 17:30:36,950
90,pay_000000005b,INV-2025-000089,2025-01-10 17:14:38,3498
129,pay_0000000082,INV-2025-000129,2025-01-02 00:29:53,1350
149,pay_0000000096,INV-2025-000148,2025-01-03 00:46:26,2999
208,pay_00000000d1,INV-2025-000209,2025-01-10 15:22:02,650


### Bonus: Additional Insights (Revenue Discrepancies)

In [16]:
# Find cases where an invoice was paid, but the payment amount does not match the invoice amount
matched = merged_df[merged_df['_merge'] == 'both']
mismatched_amounts = matched[matched['amount_rupees_inv'] != matched['amount_rupees_pay']]

print(f"Invoices with mismatched payment amounts: {len(mismatched_amounts)}")
if len(mismatched_amounts) > 0:
    display(mismatched_amounts[['invoice_id', 'amount_rupees_inv', 'amount_rupees_pay']].head())

Invoices with mismatched payment amounts: 2004


,invoice_id,amount_rupees_inv,amount_rupees_pay
510,INV-2025-000506,900,3899.0
684,INV-2025-000680,550,1849.0
1235,INV-2025-001228,880,1300.0
1305,INV-2025-001298,550,120.0
1429,INV-2025-001421,1400,2100.0
